# Quantization Comparison — Baseline vs Static INT8 vs DQNet

Compares accuracy, inference latency (averaged over 30 runs), and FLOPs across:
- **Baseline** — FP32 ResNet-20
- **Static INT8** — PyTorch built-in post-training static quantization
- **DQNet** — Dynamic per-image bit-width selection


In [ ]:
!git clone https://github.com/MrRoboto11102003/Quantization_project.git 2>/dev/null; true
!pip install thop -q
import sys
sys.path.append('Quantization_project')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.quantization as tq
import copy, time
import numpy as np
import matplotlib.pyplot as plt
from thop import profile
from resnet import resnet20, BasicBlock
from DQ_resnet import dq_resnet20, compute_bitops, max_bitops, get_mean_bits, BIT_OPTIONS

In [ ]:
import torchvision
import torchvision.transforms as transforms

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

calibration_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform_test)
calibration_loader = torch.utils.data.DataLoader(calibration_set, batch_size=128, shuffle=False, num_workers=2)

## Helper Functions

In [ ]:
def evaluate(model, loader, device='cpu'):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            out = model(inputs)
            if isinstance(out, tuple):
                out = out[0]
            _, predicted = out.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return 100. * correct / total

def measure_latency(model, loader, device='cpu', runs=30):
    model.eval()
    with torch.no_grad():
        for inputs, _ in loader:
            inputs = inputs.to(device)
            out = model(inputs)
        if torch.cuda.is_available() and device != 'cpu':
            torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(runs):
            if torch.cuda.is_available() and device != 'cpu':
                torch.cuda.synchronize()
            start = time.time()
            for inputs, _ in loader:
                inputs = inputs.to(device)
                _ = model(inputs)
            if torch.cuda.is_available() and device != 'cpu':
                torch.cuda.synchronize()
            times.append(time.time() - start)
    return np.mean(times), np.std(times)

## 1 — Baseline FP32

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BASELINE_PATH = 'Quantization_project/resnet20_base_best.pth'
net_base = resnet20()
net_base.load_state_dict(torch.load(BASELINE_PATH, map_location='cpu'))
net_base = net_base.to(device)
net_base.eval()

base_acc = evaluate(net_base, testloader, device)
base_lat, base_std = measure_latency(net_base, testloader, device)

dummy = torch.randn(1, 3, 32, 32).to(device)
base_flops, _ = profile(net_base, inputs=(dummy,), verbose=False)

print(f'Baseline  — Acc: {base_acc:.2f}%  Latency: {base_lat:.4f}±{base_std:.4f}s  FLOPs: {base_flops/1e6:.2f}M')

## 2 — Static INT8 Quantization (PyTorch built-in)

PyTorch's built-in static quantization requires CPU to run efficiently (`fbgemm` or `qnnpack` backends). Using CUDA will crash because quantized ops are generally CPU-only in core PyTorch. 

Furthermore, we must rewrite `BasicBlock` to explicitly use `QuantStub` and `DeQuantStub` to map inputs to quantized domain, and `FloatFunctional` to handle residual addition safely without crashing the backend.

In [ ]:
class QuantizedBasicBlock(nn.Module):
    expansion = 1

    def __init__(self, in_planes, planes, stride=1):
        super(QuantizedBasicBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )
        self.skip_add = torch.nn.quantized.FloatFunctional()
        self.relu1 = nn.ReLU()
        self.relu2 = nn.ReLU()

    def forward(self, x):
        out = self.relu1(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.skip_add.add(out, self.shortcut(x))
        out = self.relu2(out)
        return out

class QuantizedResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=10):
        super(QuantizedResNet, self).__init__()
        self.in_planes = 16
        
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16)
        self.relu = nn.ReLU()
        self.layer1 = self._make_layer(block, 16, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 32, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 64, num_blocks[2], stride=2)
        self.linear = nn.Linear(64, num_classes)

    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for s in strides:
            layers.append(block(self.in_planes, planes, s))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.quant(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        out = self.linear(out)
        out = self.dequant(out)
        return out

net_int8 = QuantizedResNet(QuantizedBasicBlock, [3, 3, 3])
net_int8.load_state_dict(torch.load(BASELINE_PATH, map_location='cpu'), strict=False)
net_int8.eval()

# PyTorch native static quantization relies on fbgemm (x86)
net_int8.qconfig = tq.get_default_qconfig('fbgemm')
net_int8_prepared = tq.prepare(net_int8, inplace=False)

print('Calibrating...')
with torch.no_grad():
    for i, (inputs, _) in enumerate(calibration_loader):
        net_int8_prepared(inputs)
        if i >= 50:
            break

net_int8_q = tq.convert(net_int8_prepared, inplace=False)
print('Static INT8 quantization complete.')

int8_acc = evaluate(net_int8_q, testloader, 'cpu')
int8_lat, int8_std = measure_latency(net_int8_q, testloader, 'cpu')

int8_flops = base_flops / 4.0

print(f'Static INT8 — Acc: {int8_acc:.2f}%  Latency: {int8_lat:.4f}±{int8_std:.4f}s  FLOPs(eff): {int8_flops/1e6:.2f}M')

## 3 — DQNet (Dynamic Quantization)

In [ ]:
DQ_PATH = 'Quantization_project/dq_resnet20 (2).pth'

dq_model = dq_resnet20(bit_options=BIT_OPTIONS).to(device)
dq_model.load_state_dict(torch.load(DQ_PATH, map_location=device))
dq_model.eval()

dq_acc = evaluate(dq_model, testloader, device)
dq_lat, dq_std = measure_latency(dq_model, testloader, device)

fp32_max = max_bitops(dq_model.layer_flops)
bitops_ratios = []
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _, soft_bits_list = dq_model(inputs)
        bitops_ratios.append(compute_bitops(soft_bits_list, dq_model.layer_flops).item() / fp32_max)

avg_bitops_ratio = np.mean(bitops_ratios)
dq_effective_flops = base_flops * avg_bitops_ratio

all_bits = []
with torch.no_grad():
    for inputs, _ in testloader:
        inputs = inputs.to(device)
        _, soft_bits_list = dq_model(inputs)
        mb = get_mean_bits(soft_bits_list)
        all_bits.append(mb.cpu().numpy())
all_bits = np.concatenate(all_bits, axis=0)
overall_mean_bits = all_bits.mean()

print(f'DQNet     — Acc: {dq_acc:.2f}%  Latency: {dq_lat:.4f}±{dq_std:.4f}s  Eff. FLOPs: {dq_effective_flops/1e6:.2f}M')
print(f'            BitOPs ratio: {avg_bitops_ratio:.4f}  Mean bits: {overall_mean_bits:.2f}')

## Summary Table

In [ ]:
results = {
    'Baseline (FP32)':  {'acc': base_acc,  'lat': base_lat,  'std': base_std,  'flops': base_flops},
    'Static INT8':      {'acc': int8_acc,  'lat': int8_lat,  'std': int8_std,  'flops': int8_flops},
    'DQNet':            {'acc': dq_acc,    'lat': dq_lat,    'std': dq_std,    'flops': dq_effective_flops},
}

print(f'{"Model":<18} {"Accuracy (%)":>14} {"Latency (s)":>16} {"FLOPs (M)":>12}')
print('-' * 64)
for name, r in results.items():
    print(f'{name:<18} {r["acc"]:>13.2f}% {r["lat"]:>10.4f}±{r["std"]:.4f} {r["flops"]/1e6:>11.2f}')

## Bar Charts

In [ ]:
names = list(results.keys())
accs  = [results[n]['acc']  for n in names]
lats  = [results[n]['lat']  for n in names]
stds  = [results[n]['std']  for n in names]
flops = [results[n]['flops'] / 1e6 for n in names]

colors = ['#2196F3', '#4CAF50', '#FF9800']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bars = axes[0].bar(names, accs, color=colors)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Test Accuracy')
axes[0].set_ylim(min(accs) - 5, max(accs) + 3)
for bar, v in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.3, f'{v:.2f}%', ha='center', fontweight='bold')

bars = axes[1].bar(names, lats, yerr=stds, color=colors, capsize=5)
axes[1].set_ylabel('Inference Time (s)')
axes[1].set_title('Avg Inference Latency (30 runs)')
for bar, v in zip(bars, lats):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + max(stds)*0.5, f'{v:.3f}s', ha='center', fontweight='bold')

bars = axes[2].bar(names, flops, color=colors)
axes[2].set_ylabel('FLOPs (M)')
axes[2].set_title('Effective FLOPs')
for bar, v in zip(bars, flops):
    axes[2].text(bar.get_x() + bar.get_width()/2, v + max(flops)*0.02, f'{v:.2f}M', ha='center', fontweight='bold')

for ax in axes:
    ax.grid(True, alpha=0.3, axis='y')
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('comparison_results.png', dpi=150, bbox_inches='tight')
plt.show()

## Normalized Comparison (Relative to Baseline)

In [ ]:
metrics = ['Accuracy', 'Latency', 'FLOPs']
baseline_vals = [base_acc, base_lat, base_flops / 1e6]

norm_data = {}
for name in names:
    r = results[name]
    norm_data[name] = [
        r['acc'] / base_acc * 100,
        r['lat'] / base_lat * 100,
        r['flops'] / base_flops * 100,
    ]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
for i, (name, vals) in enumerate(norm_data.items()):
    bars = ax.bar(x + i * width, vals, width, label=name, color=colors[i])
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{v:.1f}%', ha='center', fontsize=9)

ax.axhline(y=100, color='gray', linestyle='--', alpha=0.5, label='Baseline (100%)')
ax.set_ylabel('% of Baseline')
ax.set_title('Normalized Comparison (Baseline = 100%)')
ax.set_xticks(x + width)
ax.set_xticklabels(metrics)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('comparison_normalized.png', dpi=150, bbox_inches='tight')
plt.show()